# Causal Masks for Reward Redistribution: A Tutorial

**Based on**: *Interpretable Reward Redistribution in Reinforcement Learning: A Causal Approach* (Zhang et al., NeurIPS 2023)

---

In many RL problems the reward signal is delayed: you observe only the total return at the end of an episode,
not the per-step reward that caused it. Reward redistribution methods recover a proxy per-step reward
$\hat{r}_t$ such that $\sum_t \gamma^{t-1} \hat{r}_t = R$, giving the policy a dense training signal.

GRD (Generative Return Decomposition) goes a step further. Instead of just fitting the return constraint,
it asks: *which dimensions of the state and action actually matter for the reward?*
It answers this question by learning a **causal mask** — a set of binary vectors that select only the
causally relevant inputs to the reward function.

This notebook explains, with runnable code:

1. The causal model and what the masks represent
2. The Bayesian treatment of the masks (Bernoulli prior, regularised posterior)
3. How discrete masks are made differentiable via Gumbel-Softmax
4. The three loss functions and the joint training loop
5. How to extract the compact state representation $s^{\min}$
6. Challenges, limitations, and complexity
7. Practical simplifications

The examples use small synthetic MDPs so every concept runs in seconds on CPU.

In [1]:
import collections
import itertools

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

torch.manual_seed(0)
np.random.seed(0)

# Compact plot style
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

---
## 1. The Causal Model

### 1.1 Factored MDP assumption

GRD assumes the state $s_t \in \mathbb{R}^{d_s}$ and action $a_t \in \mathbb{R}^{d_a}$ decompose into
**independent scalar dimensions**. Two separate causal graphs govern the system:

```
Dynamics graph (how the world evolves)
  s_{t-1}, a_{t-1}  ──►  s_{i,t}    for each state dimension i

Reward graph (what drives the reward)
  s_t, a_t          ──►  r_t
```

Not every dimension participates in every edge. The edges that *exist* are encoded as binary masks:

| Mask | Shape | Meaning |
|------|-------|---------|
| $C^{s\to s}$ | $d_s \times d_s$ | state dim $j$ causes next-state dim $i$ |
| $C^{a\to s}$ | $d_a \times d_s$ | action dim $j$ causes next-state dim $i$ |
| $C^{s\to r}$ | $d_s$ | state dim $j$ causes the reward |
| $C^{a\to r}$ | $d_a$ | action dim $j$ causes the reward |

The generative equations become:

$$s_{i,t+1} = f\!\left(C^{s\to s}_{\cdot,i} \odot s_t,\; C^{a\to s}_{\cdot,i} \odot a_t,\; \epsilon_{s,i}\right)$$

$$r_t = g\!\left(C^{s\to r} \odot s_t,\; C^{a\to r} \odot a_t,\; \epsilon_r\right)$$

The $\odot$ (elementwise multiply) zeros out dimensions that have no causal edge — those dimensions
are simply ignored by $f$ and $g$.

### 1.2 Synthetic example

Let us build a toy 4-dim state / 2-dim action MDP where the **ground-truth reward uses only
$s_0$ and $a_1$**, and the remaining dimensions are pure noise.

In [2]:
STATE_DIM = 4
ACTION_DIM = 2
SEQ_LEN = 10        # steps per episode
N_EPISODES = 500

# Ground-truth masks (what the model should recover)
TRUE_C_SR = torch.tensor([1, 0, 0, 0], dtype=torch.float32)  # only s[0] affects r
TRUE_C_AR = torch.tensor([0, 1],        dtype=torch.float32)  # only a[1] affects r


def true_reward(state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
    """Ground-truth Markovian reward: linear in s[0] and a[1] only."""
    return 2.0 * state[..., 0] + 3.0 * action[..., 1]


def collect_dataset(
    n_episodes: int, seq_len: int
) -> tuple:
    """Collect trajectories from the toy MDP."""
    states  = torch.randn(n_episodes, seq_len, STATE_DIM)
    actions = torch.randn(n_episodes, seq_len, ACTION_DIM)
    rewards = true_reward(states, actions)           # (episodes, seq_len)
    returns = rewards.sum(dim=1)                     # undiscounted episode return
    return states, actions, rewards, returns


states, actions, rewards, returns = collect_dataset(N_EPISODES, SEQ_LEN)
print(f"states  : {states.shape}")
print(f"actions : {actions.shape}")
print(f"rewards : {rewards.shape}  (hidden from the learner)")
print(f"returns : {returns.shape}   (the only supervision signal)")
print(f"\nReturn range: [{returns.min():.1f}, {returns.max():.1f}]")

states  : torch.Size([500, 10, 4])
actions : torch.Size([500, 10, 2])
rewards : torch.Size([500, 10])  (hidden from the learner)
returns : torch.Size([500])   (the only supervision signal)

Return range: [-36.7, 27.5]


---
## 2. Bayesian Treatment of the Masks

### 2.1 Masks as Bernoulli random variables

Each entry $C_{ij} \in \{0, 1\}$ of a mask is treated as a **Bernoulli random variable**:

$$C_{ij} \sim \text{Bernoulli}(p_{ij}), \qquad p_{ij} = \sigma(\psi_{ij})$$

The scalar logit $\psi_{ij}$ is a free parameter (no input — the mask is global, not conditioned
on the state). GRD represents each logit as a **2-element vector** $[\psi^0, \psi^1]$ and takes
$p_{ij} = \text{softmax}([\psi^0, \psi^1])[1]$, which is equivalent to a Bernoulli but symmetrises
the parameterisation for Gumbel-Softmax (covered in §3).

### 2.2 The prior and the regularisation loss

Without any preference, every edge exists with probability 0.5. The prior encodes a sparsity belief:
most edges *should not* exist. This is implemented as a **cross-entropy** (KL) loss that pulls each
edge probability toward zero:

$$L_{\text{reg}} = \lambda \sum_{ij} \underbrace{-\log(1 - p_{ij})}_{\text{CE with target 0}}$$

Minimising $L_{\text{reg}}$ drives $p_{ij} \to 0$ (no edge). The data losses $L_{\text{rew}}$ and
$L_{\text{dyn}}$ push back, keeping edges that genuinely explain the observations.

The MAP (maximum-a-posteriori) estimate of the masks is the solution that balances these two forces:

> **Keep an edge if and only if removing it makes the data significantly harder to explain.**

Different $\lambda$ values weight the prior more or less strongly:

In [ ]:
class CausalStructure(nn.Module):
    """Learnable Bernoulli parameters for all four causal masks.

    Each mask entry is parameterised by a 2-element logit vector [ψ⁰, ψ¹];
    p(edge=1) = softmax([ψ⁰, ψ¹])[1].
    """

    def __init__(self, state_dim: int, action_dim: int) -> None:
        super().__init__()
        self.state_dim  = state_dim
        self.action_dim = action_dim

        # Shape: (from_dim, to_dim, 2)  — 2 logits per entry
        self.psi_ss = nn.Parameter(torch.zeros(state_dim,  state_dim,  2))  # s → s
        self.psi_as = nn.Parameter(torch.zeros(action_dim, state_dim,  2))  # a → s
        self.psi_sr = nn.Parameter(torch.zeros(state_dim,  2))              # s → r
        self.psi_ar = nn.Parameter(torch.zeros(action_dim, 2))              # a → r

    def edge_probabilities(self) -> dict:
        """Return p(edge=1) for every mask, shape matches the mask."""
        return {
            "ss": F.softmax(self.psi_ss, dim=-1)[..., 1],   # (d_s, d_s)
            "as": F.softmax(self.psi_as, dim=-1)[..., 1],   # (d_a, d_s)
            "sr": F.softmax(self.psi_sr, dim=-1)[..., 1],   # (d_s,)
            "ar": F.softmax(self.psi_ar, dim=-1)[..., 1],   # (d_a,)
        }


def regularisation_loss(
    causal: CausalStructure,
    lambdas: dict,
) -> torch.Tensor:
    """Sparsity prior: cross-entropy pushing each p(edge) toward zero."""
    probs = causal.edge_probabilities()
    eps = 1e-8
    # -log(1 - p) penalises edges that are likely to exist
    loss  = lambdas["sr"] * (-torch.log(1 - probs["sr"] + eps)).sum()
    loss += lambdas["ar"] * (-torch.log(1 - probs["ar"] + eps)).sum()
    loss += lambdas["ss"] * (-torch.log(1 - probs["ss"] + eps)).sum()
    loss += lambdas["as"] * (-torch.log(1 - probs["as"] + eps)).sum()
    return loss


# Demonstrate the effect of λ on prior strength
causal = CausalStructure(STATE_DIM, ACTION_DIM)
lambdas_demo = {"sr": 1e-4, "ar": 1e-4, "ss": 1e-6, "as": 1e-6}
loss_val = regularisation_loss(causal, lambdas_demo)
print(f"Regularisation loss at initialisation (all p=0.5): {loss_val.item():.4f}")

# After forcing all probs to near-zero the loss should drop significantly
with torch.no_grad():
    causal.psi_sr.data = torch.tensor([[ 5.0, -5.0], [ 5.0, -5.0],
                                        [ 5.0, -5.0], [ 5.0, -5.0]])
    causal.psi_ar.data = torch.tensor([[ 5.0, -5.0], [ 5.0, -5.0]])
print(f"Regularisation loss when all reward edges suppressed : {regularisation_loss(causal, lambdas_demo).item():.6f}")

# Reset
causal = CausalStructure(STATE_DIM, ACTION_DIM)

---
## 3. Making Discrete Masks Differentiable: Gumbel-Softmax

The masks $C_{ij} \in \{0,1\}$ are discrete — the gradient of the loss with respect to a hard
binary variable is zero almost everywhere. Backpropagation cannot update $\psi_{ij}$.

**The Gumbel-Softmax trick** replaces the hard sample with a soft, temperature-controlled approximation:

$$\tilde{C}_{ij} = \frac{\exp\!\left(\frac{\log p_{ij} + G_1}{\tau}\right)}{\exp\!\left(\frac{\log p_{ij} + G_1}{\tau}\right) + \exp\!\left(\frac{\log(1-p_{ij}) + G_0}{\tau}\right)}$$

where $G_k \sim \text{Gumbel}(0,1)$ are i.i.d. noise samples and $\tau > 0$ is a temperature:

- **$\tau \to 0$**: $\tilde{C}_{ij}$ collapses to a hard 0/1 (unbiased but no gradient)
- **$\tau \to \infty$**: $\tilde{C}_{ij} \to 0.5$ for all entries (smooth but biased)
- **$\tau \approx 0.5$–$2$**: Good practical balance

At **inference time**, a hard greedy selection is used:

$$C_{ij} = \mathbb{1}[\psi^1_{ij} \geq \psi^0_{ij}]$$

PyTorch provides `F.gumbel_softmax` directly. Let us visualise how the temperature controls
the sharpness of the samples:

In [ ]:
def sample_mask_gumbel(
    psi: torch.Tensor,
    tau: float = 1.0,
    hard: bool = False,
) -> torch.Tensor:
    """Sample a soft (or hard) binary mask from logit parameters psi.

    Args:
        psi: Logit tensor of shape (..., 2), last dim is [logit_0, logit_1].
        tau: Gumbel-Softmax temperature.
        hard: If True, return hard 0/1 via straight-through estimator.

    Returns:
        Mask entries in [0, 1], same shape as psi[..., 0].
    """
    soft = F.gumbel_softmax(psi, tau=tau, hard=hard, dim=-1)
    return soft[..., 1]   # probability of edge=1


def greedy_mask(psi: torch.Tensor) -> torch.Tensor:
    """Hard 0/1 mask: 1 iff logit_1 >= logit_0."""
    return (psi[..., 1] >= psi[..., 0]).float()


# Demonstrate temperature effect on a single Bernoulli with p=0.7
logit_p07 = torch.tensor([np.log(0.3), np.log(0.7)])

fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
temperatures = [0.1, 1.0, 5.0]
for ax, tau in zip(axes, temperatures):
    samples = [
        F.gumbel_softmax(logit_p07.unsqueeze(0), tau=tau, dim=-1)[0, 1].item()
        for _ in range(2000)
    ]
    ax.hist(samples, bins=40, range=(0, 1), color="steelblue", edgecolor="white")
    ax.set_title(f"τ = {tau}")
    ax.set_xlabel("Soft mask value")
axes[0].set_ylabel("Count")
fig.suptitle("Gumbel-Softmax samples (true p=0.7) at different temperatures", y=1.02)
plt.tight_layout()
plt.show()

print("\nAt τ=0.1 (low), samples cluster near 0 or 1 — almost hard.")
print("At τ=5.0 (high), samples cluster near 0.5 — smooth but biased.")

---
## 4. The Reward Function and the Reward Loss

Given a soft mask $\tilde{C}^{s\to r}$ and $\tilde{C}^{a\to r}$, the **reward module** $\phi_{\text{rew}}$
predicts per-step rewards from masked inputs:

$$\hat{r}_t = \phi_{\text{rew}}\!\left(\tilde{C}^{s\to r} \odot s_t,\; \tilde{C}^{a\to r} \odot a_t\right)$$

The **reward loss** constrains the sum of predicted rewards to match the observed episode return:

$$L_{\text{rew}} = \mathbb{E}_{\tau \sim \mathcal{D}}\!\left[\left\|R - \sum_{t=1}^{T} \gamma^{t-1} \hat{r}_t\right\|^2\right]$$

Note what is *not* supervised: the individual $\hat{r}_t$ values. The only constraint is that their
**sum equals the return**. The model is free to distribute credit across timesteps however it likes —
which is exactly the reward redistribution problem.

In [ ]:
class RewardModule(nn.Module):
    """MLP reward predictor operating on causally masked (state, action)."""

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self,
        state: torch.Tensor,    # (batch, seq_len, d_s)
        action: torch.Tensor,   # (batch, seq_len, d_a)
        mask_sr: torch.Tensor,  # (d_s,)  — soft or hard
        mask_ar: torch.Tensor,  # (d_a,)  — soft or hard
    ) -> torch.Tensor:          # (batch, seq_len)
        masked_s = state  * mask_sr          # zero out non-causal state dims
        masked_a = action * mask_ar          # zero out non-causal action dims
        inp = torch.cat([masked_s, masked_a], dim=-1)
        return self.net(inp).squeeze(-1)     # (batch, seq_len)


def reward_loss(
    rew_module: RewardModule,
    causal: CausalStructure,
    states: torch.Tensor,    # (batch, T, d_s)
    actions: torch.Tensor,   # (batch, T, d_a)
    returns: torch.Tensor,   # (batch,)
    tau: float = 1.0,
    gamma: float = 1.0,
) -> torch.Tensor:
    """MSE between predicted and observed episode return."""
    mask_sr = sample_mask_gumbel(causal.psi_sr, tau=tau)
    mask_ar = sample_mask_gumbel(causal.psi_ar, tau=tau)

    # Per-step predicted rewards: (batch, T)
    r_hat = rew_module(states, actions, mask_sr, mask_ar)

    # Discount weights: γ^0, γ^1, ..., γ^{T-1}
    seq_len = states.shape[1]
    discounts = torch.tensor(
        [gamma**idx for idx in range(seq_len)], dtype=torch.float32
    )

    predicted_return = (r_hat * discounts).sum(dim=1)  # (batch,)
    return F.mse_loss(predicted_return, returns)


# Quick sanity check: untrained model should have high loss
rew_module = RewardModule(STATE_DIM, ACTION_DIM)
causal     = CausalStructure(STATE_DIM, ACTION_DIM)
loss_val   = reward_loss(rew_module, causal, states[:16], actions[:16], returns[:16])
print(f"Reward loss (random init): {loss_val.item():.2f}")

---
## 5. The Dynamics Module and the Dynamics Loss

The dynamics module $\phi_{\text{dyn}}$ predicts the next state given masked inputs. Its purpose is
**not** to assist the policy directly — it trains the *transition masks* $C^{s\to s}$ and $C^{a\to s}$,
which are then used to compute $s^{\min}$ (§7).

GRD uses a **Mixture Density Network** (MDN) to allow multimodal next-state distributions:

$$P(s_{i,t+1} \mid s_t, a_t) = \sum_k \pi_k \cdot \mathcal{N}(\mu_k, \sigma_k^2)$$

The loss is the negative log-likelihood:

$$L_{\text{dyn}} = -\mathbb{E}\!\left[\sum_i \log P\!\left(s_{i,t+1} \mid C^{s\to s}_{\cdot,i} \odot s_t,\; C^{a\to s}_{\cdot,i} \odot a_t\right)\right]$$

For a tutorial-scale example we use a single Gaussian (MDN with 1 component = plain regression):

In [ ]:
class DynamicsModule(nn.Module):
    """Simple Gaussian dynamics model (single-component MDN for clarity)."""

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 64) -> None:
        super().__init__()
        # Shared trunk; outputs mean and log-std for each state dimension
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mean_head    = nn.Linear(hidden_dim, state_dim)
        self.log_std_head = nn.Linear(hidden_dim, state_dim)

    def forward(
        self,
        state: torch.Tensor,    # (batch, d_s)
        action: torch.Tensor,   # (batch, d_a)
        mask_ss: torch.Tensor,  # (d_s, d_s)
        mask_as: torch.Tensor,  # (d_a, d_s)
    ) -> tuple:
        # Apply masks: for each target dim i, zero out non-causal source dims
        # Effective input for dim i: [mask_ss[:,i]*s, mask_as[:,i]*a]
        # We apply the union mask (OR over target dims) for a shared trunk
        union_sr = (mask_ss.sum(dim=1) > 0).float()   # (d_s,)
        union_ar = (mask_as.sum(dim=1) > 0).float()   # (d_a,)
        masked_s = state  * union_sr
        masked_a = action * union_ar
        inp = torch.cat([masked_s, masked_a], dim=-1)
        hidden = self.net(inp)
        mean    = self.mean_head(hidden)
        log_std = self.log_std_head(hidden).clamp(-4, 2)
        return mean, log_std.exp()


def dynamics_loss(
    dyn_module: DynamicsModule,
    causal: CausalStructure,
    states: torch.Tensor,       # (batch, d_s)   — current state
    actions: torch.Tensor,      # (batch, d_a)
    next_states: torch.Tensor,  # (batch, d_s)
    tau: float = 1.0,
) -> torch.Tensor:
    """Gaussian NLL for next-state prediction."""
    mask_ss = sample_mask_gumbel(causal.psi_ss, tau=tau)  # (d_s, d_s)
    mask_as = sample_mask_gumbel(causal.psi_as, tau=tau)  # (d_a, d_s)

    mean, std = dyn_module(states, actions, mask_ss, mask_as)
    dist = torch.distributions.Normal(mean, std)
    return -dist.log_prob(next_states).mean()


dyn_module = DynamicsModule(STATE_DIM, ACTION_DIM)

# Use consecutive steps within an episode as (s_t, a_t, s_{t+1}) triples
s_t  = states[:, :-1, :].reshape(-1, STATE_DIM)
a_t  = actions[:, :-1, :].reshape(-1, ACTION_DIM)
s_t1 = states[:, 1:,  :].reshape(-1, STATE_DIM)

loss_dyn = dynamics_loss(dyn_module, causal, s_t[:64], a_t[:64], s_t1[:64])
print(f"Dynamics NLL (random init): {loss_dyn.item():.2f}")

---
## 6. The Joint Training Loop

All three losses are minimised simultaneously:

$$\min_{\phi_{\text{cau}}, \phi_{\text{rew}}, \phi_{\text{dyn}}} \; L_{\text{rew}} + L_{\text{dyn}} + L_{\text{reg}}$$

In GRD the reward and dynamics updates use **separate minibatches** (D1 = full trajectories;
D2 = transitions), but the causal parameters $\phi_{\text{cau}}$ receive gradients from all three.

For clarity, our toy loop runs a single shared Adam step over all parameters at each iteration.

In [ ]:
def train(
    states: torch.Tensor,
    actions: torch.Tensor,
    returns: torch.Tensor,
    n_steps: int = 2000,
    batch_size: int = 32,
    tau: float = 1.0,
    lambdas: dict | None = None,
) -> dict:
    """Train GRD causal structure + reward module on the toy MDP."""
    if lambdas is None:
        lambdas = {"sr": 5e-3, "ar": 5e-3, "ss": 1e-5, "as": 1e-5}

    rew_mod  = RewardModule(STATE_DIM, ACTION_DIM)
    dyn_mod  = DynamicsModule(STATE_DIM, ACTION_DIM)
    causal   = CausalStructure(STATE_DIM, ACTION_DIM)
    opt      = optim.Adam(
        list(rew_mod.parameters()) +
        list(dyn_mod.parameters()) +
        list(causal.parameters()),
        lr=3e-4,
    )

    n_episodes = states.shape[0]
    history    = collections.defaultdict(list)

    # Pre-flatten transition triples for dynamics batching
    s_t_all  = states[:, :-1, :].reshape(-1, STATE_DIM)
    a_t_all  = actions[:, :-1, :].reshape(-1, ACTION_DIM)
    s_t1_all = states[:, 1:, :].reshape(-1, STATE_DIM)

    for step_idx in range(n_steps):
        opt.zero_grad()

        # --- Reward loss: full trajectories ---
        ep_idx = torch.randint(n_episodes, (batch_size,))
        l_rew  = reward_loss(
            rew_mod, causal,
            states[ep_idx], actions[ep_idx], returns[ep_idx],
            tau=tau,
        )

        # --- Dynamics loss: individual transitions ---
        tr_idx = torch.randint(len(s_t_all), (batch_size,))
        l_dyn  = dynamics_loss(
            dyn_mod, causal,
            s_t_all[tr_idx], a_t_all[tr_idx], s_t1_all[tr_idx],
            tau=tau,
        )

        # --- Regularisation: sparsity prior on all masks ---
        l_reg  = regularisation_loss(causal, lambdas)

        loss = l_rew + l_dyn + l_reg
        loss.backward()
        opt.step()

        if step_idx % 200 == 0:
            history["step"].append(step_idx)
            history["l_rew"].append(l_rew.item())
            history["l_dyn"].append(l_dyn.item())
            history["l_reg"].append(l_reg.item())
            probs = causal.edge_probabilities()
            history["p_sr"].append(probs["sr"].detach().tolist())
            history["p_ar"].append(probs["ar"].detach().tolist())

    return history, causal, rew_mod


print("Training GRD on the toy MDP...")
history, causal_trained, rew_trained = train(states, actions, returns)
print("Done.")

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 3, figsize=(13, 3))

axes[0].plot(history["step"], history["l_rew"], color="steelblue")
axes[0].set_title("Reward loss $L_{rew}$")
axes[0].set_xlabel("Step")

axes[1].plot(history["step"], history["l_dyn"], color="darkorange")
axes[1].set_title("Dynamics loss $L_{dyn}$")
axes[1].set_xlabel("Step")

axes[2].plot(history["step"], history["l_reg"], color="seagreen")
axes[2].set_title("Regularisation loss $L_{reg}$")
axes[2].set_xlabel("Step")

plt.tight_layout()
plt.show()

In [ ]:
# Visualise the learned edge probabilities for the reward masks
probs = causal_trained.edge_probabilities()
p_sr  = probs["sr"].detach().numpy()
p_ar  = probs["ar"].detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(9, 3))

axes[0].bar(range(STATE_DIM), p_sr, color="steelblue", edgecolor="white")
axes[0].axhline(0.5, color="red", linestyle="--", label="decision boundary")
axes[0].set_xticks(range(STATE_DIM))
axes[0].set_xticklabels([f"s[{idx}]" for idx in range(STATE_DIM)])
axes[0].set_ylim(0, 1)
axes[0].set_title(r"$p(C^{s\to r}_i = 1)$")
axes[0].legend()

# Ground truth: s[0]=1, others=0
axes[0].scatter(
    range(STATE_DIM),
    TRUE_C_SR.numpy(),
    color="gold", marker="*", s=180, zorder=5, label="ground truth"
)
axes[0].legend()

axes[1].bar(range(ACTION_DIM), p_ar, color="darkorange", edgecolor="white")
axes[1].axhline(0.5, color="red", linestyle="--")
axes[1].set_xticks(range(ACTION_DIM))
axes[1].set_xticklabels([f"a[{idx}]" for idx in range(ACTION_DIM)])
axes[1].set_ylim(0, 1)
axes[1].set_title(r"$p(C^{a\to r}_j = 1)$")
axes[1].scatter(
    range(ACTION_DIM),
    TRUE_C_AR.numpy(),
    color="gold", marker="*", s=180, zorder=5, label="ground truth"
)
axes[1].legend()

plt.suptitle("Learned vs. ground-truth reward masks", y=1.02)
plt.tight_layout()
plt.show()

# Hard masks
c_sr_hard = greedy_mask(causal_trained.psi_sr)
c_ar_hard = greedy_mask(causal_trained.psi_ar)
print(f"Hard C^{{s→r}} (learned) : {c_sr_hard.int().tolist()}")
print(f"Hard C^{{s→r}} (true)    : {TRUE_C_SR.int().tolist()}")
print(f"Hard C^{{a→r}} (learned) : {c_ar_hard.int().tolist()}")
print(f"Hard C^{{a→r}} (true)    : {TRUE_C_AR.int().tolist()}")

---
## 7. Compact State Representation $s^{\min}$

Once the masks have converged, GRD computes the **minimal sufficient state** for the policy.
The idea: a dimension matters for control if it either

1. **directly** affects the reward ($C^{s\to r}_i = 1$), or
2. **transitively** affects a reward-relevant dimension via the dynamics ($C^{s\to s}_{j,i}=1$
   for some $j \in s^{\min}$).

Formally:

$$C^{s\to\pi} = C^{s\to r} \vee \bigvee_{i\,:\,C^{s\to r}_i=1} C^{s\to s}_{\cdot, i}$$

This is computed by a simple BFS/OR traversal over the causal graph:

In [ ]:
def compact_representation_mask(
    c_sr: torch.Tensor,   # (d_s,)    — hard binary
    c_ss: torch.Tensor,   # (d_s, d_s) — hard binary, c_ss[j, i] = j causes i
) -> torch.Tensor:
    """Compute C^{s→π}: state dimensions with direct or transitive reward influence.

    Iterates until no new dimension is added (fixed-point of the OR-closure).
    """
    relevant = c_sr.clone()          # start with direct parents of r
    while True:
        # Any state dim j that feeds into a currently relevant dim
        # c_ss[:, i] = sources of dim i; if i is relevant, include its sources
        new = (c_ss * relevant.unsqueeze(0)).sum(dim=1).clamp(max=1)
        updated = (relevant + new).clamp(max=1)
        if torch.equal(updated, relevant):
            break
        relevant = updated
    return relevant


# In our toy MDP the transition is independent — no state-to-state edges expected
c_sr_hard = greedy_mask(causal_trained.psi_sr)
c_ss_hard = greedy_mask(causal_trained.psi_ss)
c_s_pi    = compact_representation_mask(c_sr_hard, c_ss_hard)

print(f"C^{{s→r}}   (direct reward parents) : {c_sr_hard.int().tolist()}")
print(f"C^{{s→π}}   (compact rep. mask)     : {c_s_pi.int().tolist()}")
print(f"\nPolicy input dimensionality: {int(c_s_pi.sum())} / {STATE_DIM}")
print(f"State dimensions dropped   : {STATE_DIM - int(c_s_pi.sum())}")

The compact mask is then applied at policy training time:

```python
s_min = state * c_s_pi     # zero out irrelevant dims before feeding the actor/critic
```

**Why this helps**: the policy's input dimensionality is reduced, its value landscape is simpler,
and injected noise in irrelevant dimensions (which happens in real MuJoCo) cannot mislead training.

---
## 8. Effect of the Sparsity Hyperparameter $\lambda$

The sparsity weights $\lambda$ are the most consequential hyperparameters. Let us demonstrate
how they control the recovered mask:

In [ ]:
lambda_grid = [0.0, 1e-4, 5e-3, 1e-1]
results     = {}

for lam in lambda_grid:
    lam_dict = {"sr": lam, "ar": lam, "ss": lam * 0.01, "as": lam * 0.01}
    hist, cau, rew = train(
        states, actions, returns,
        n_steps=1500, lambdas=lam_dict
    )
    results[lam] = {
        "c_sr": greedy_mask(cau.psi_sr).int().tolist(),
        "c_ar": greedy_mask(cau.psi_ar).int().tolist(),
        "l_rew_final": hist["l_rew"][-1],
    }
    print(
        f"λ={lam:.0e}  C^s→r={results[lam]['c_sr']}  "
        f"C^a→r={results[lam]['c_ar']}  "
        f"L_rew={results[lam]['l_rew_final']:.3f}"
    )

print(f"\nGround truth: C^s→r={TRUE_C_SR.int().tolist()}  C^a→r={TRUE_C_AR.int().tolist()}")

The pattern is consistent with the paper's ablation (Table 1):

| $\lambda$ | Outcome |
|-----------|-------------------------------------------|
| `0.0` | All edges retained; no compression |
| Moderate | Correct sparse structure recovered |
| Very large | True causal edges pruned; underfitting |

There is no principled formula for $\lambda$ — it requires per-task tuning.

---
## 9. Challenges and Limitations

### 9.1 Identifiability breaks down outside linear Gaussian models

The theoretical guarantee (Proposition 1) that the masks are *uniquely* recoverable relies on:
- Linear reward and transition functions
- Gaussian noise
- Weight-decay regularisation

MuJoCo uses nonlinear physics. The empirical results are good, but the theoretical story is
incomplete. In nonlinear settings there can be **spurious equivalences**: two different
mask configurations may fit the return equally well.

### 9.2 Joint learning instability

The generative model and policy share a replay buffer and train simultaneously.
Poor reward estimates early in training produce bad policy data, which in turn
degrades reward learning. The paper offers no explicit stabilisation mechanism;
practitioners must monitor both loss curves.

In [ ]:
# Demonstrate how a biased initial mask corrupts reward learning
# Scenario: force C^{s→r} to the wrong dimensions at the start

causal_biased = CausalStructure(STATE_DIM, ACTION_DIM)
with torch.no_grad():
    # Force mask to select s[1], s[2] (wrong) instead of s[0] (correct)
    causal_biased.psi_sr.data = torch.tensor([
        [ 5.0, -5.0],   # s[0]: edge=0
        [-5.0,  5.0],   # s[1]: edge=1  ← wrong
        [-5.0,  5.0],   # s[2]: edge=1  ← wrong
        [ 5.0, -5.0],   # s[3]: edge=0
    ])

rew_biased = RewardModule(STATE_DIM, ACTION_DIM)

# With the wrong mask the reward module cannot fit the return — even after many steps
opt_biased = optim.Adam(
    list(rew_biased.parameters()), lr=3e-4
)   # only train reward head, not causal

biased_losses = []
for _ in range(400):
    opt_biased.zero_grad()
    ep_idx = torch.randint(N_EPISODES, (32,))
    loss = reward_loss(
        rew_biased, causal_biased,
        states[ep_idx], actions[ep_idx], returns[ep_idx]
    )
    loss.backward()
    opt_biased.step()
    biased_losses.append(loss.item())

# Same for the correct mask
causal_correct = CausalStructure(STATE_DIM, ACTION_DIM)
with torch.no_grad():
    causal_correct.psi_sr.data = torch.tensor([
        [-5.0,  5.0], [ 5.0, -5.0], [ 5.0, -5.0], [ 5.0, -5.0]
    ])
    causal_correct.psi_ar.data = torch.tensor([[ 5.0, -5.0], [-5.0,  5.0]])

rew_correct = RewardModule(STATE_DIM, ACTION_DIM)
opt_correct = optim.Adam(list(rew_correct.parameters()), lr=3e-4)
correct_losses = []
for _ in range(400):
    opt_correct.zero_grad()
    ep_idx = torch.randint(N_EPISODES, (32,))
    loss = reward_loss(
        rew_correct, causal_correct,
        states[ep_idx], actions[ep_idx], returns[ep_idx]
    )
    loss.backward()
    opt_correct.step()
    correct_losses.append(loss.item())

plt.figure(figsize=(8, 4))
plt.semilogy(biased_losses,  label="wrong mask  (s[1], s[2])", color="crimson")
plt.semilogy(correct_losses, label="correct mask (s[0], a[1])", color="steelblue")
plt.xlabel("Training step")
plt.ylabel("Reward loss (log scale)")
plt.title("Wrong causal mask → irreducible reward loss")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Final loss — wrong mask : {np.mean(biased_losses[-50:]):.2f}")
print(f"Final loss — correct    : {np.mean(correct_losses[-50:]):.2f}")

### 9.3 Stationarity assumption

GRD assumes **one fixed causal graph** for the entire episode. In non-stationary environments
(e.g., contact-rich manipulation where relevant dimensions change mid-episode) this is violated.
The paper acknowledges this as a limitation.

### 9.4 No unobserved confounders

If a hidden variable $Z$ causes both a state dimension and the reward (a confounder), the
learned mask may incorrectly include that dimension's ancestors or exclude the true causal
parents. GRD provides no mechanism for detecting or handling confounders.

### 9.5 The unit-of-credit problem is not fully solved

The return constraint $\sum_t \hat{r}_t = R$ is necessary but not sufficient to recover the
true per-step reward. There are infinitely many redistribution schemes that satisfy the
constraint. The dynamics regulariser provides a *preference* (masks consistent with both reward
and transition structure), but two models with different reward redistributions can achieve
the same $L_{\text{rew}}$.

---
## 10. Resource Requirements and Complexity

### 10.1 Parameter count

In [ ]:
def count_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters())


causal_demo = CausalStructure(STATE_DIM, ACTION_DIM)
rew_demo    = RewardModule(STATE_DIM, ACTION_DIM, hidden_dim=256)
dyn_demo    = DynamicsModule(STATE_DIM, ACTION_DIM, hidden_dim=256)

print("Parameter counts (toy MDP: d_s=4, d_a=2)")
print(f"  Causal structure  : {count_parameters(causal_demo):>8,}")
print(f"  Reward module     : {count_parameters(rew_demo):>8,}")
print(f"  Dynamics module   : {count_parameters(dyn_demo):>8,}")

# Realistic MuJoCo scale
print("\nRealistic scale (HalfCheetah: d_s=17, d_a=6, hidden=256)")
for d_s, d_a, name in [(17, 6, "HalfCheetah"), (111, 8, "Ant"), (376, 17, "Humanoid")]:
    cau = CausalStructure(d_s, d_a)
    rew = RewardModule(d_s, d_a, hidden_dim=256)
    dyn = DynamicsModule(d_s, d_a, hidden_dim=256)
    total = count_parameters(cau) + count_parameters(rew) + count_parameters(dyn)
    cau_params = count_parameters(cau)
    print(
        f"  {name:<12}  causal={cau_params:>6,}  "
        f"rew={count_parameters(rew):>7,}  "
        f"dyn={count_parameters(dyn):>7,}  "
        f"total={total:>8,}"
    )

### 10.2 Computational complexity

| Operation | Cost per forward pass |
|---|---|
| Mask application (reward) | $O(d_s + d_a)$ — elementwise multiply |
| Mask application (dynamics) | $O(d_s^2 + d_a \cdot d_s)$ — per-target union |
| Gumbel-Softmax sampling | $O(d_s^2 + d_a \cdot d_s + d_s + d_a)$ — one sample per entry |
| Reward MLP forward | $O(B \cdot T \cdot d_h^2)$ — B episodes, T steps, $d_h$ hidden |
| Dynamics MLP forward | $O(B \cdot d_h^2)$ per transition |

The **causal structure** parameter count scales as $O(d_s^2 + d_a \cdot d_s)$.
For Humanoid ($d_s = 376$) this is already $\approx 150{,}000$ parameters in the mask
alone — larger than either the reward or dynamics network.

The **$O(d_s^2)$ scaling is the dominant bottleneck** for high-dimensional states.
This is manageable for MuJoCo (max 376 dims) but becomes prohibitive for visual inputs
or language-conditioned tasks.

### 10.3 Training cost

The paper reports 3×10⁵ environment steps total, with 100 mini-batch updates per 100
environment steps. Each update processes:
- D1: 4 full trajectories × T steps for the reward loss
- D2: 256 transitions for the dynamics loss

Estimated wall-clock time:
- CPU only (8 cores): ~2–4 hours per seed on HalfCheetah
- GPU (A100): ~20–30 minutes per seed

---
## 11. Practical Simplifications

If the full GRD machinery is too expensive or complex, several simplifications are available.
Each trades some theoretical rigour for reduced cost.

### 11.1 Drop the dynamics module entirely

If you only need the **reward mask** ($C^{s\to r}$, $C^{a\to r}$) and not the compact
representation, you can skip $L_{\text{dyn}}$ and $\phi_{\text{dyn}}$ altogether.
This halves the parameter count and eliminates the $O(d_s^2)$ dynamics bottleneck.

In [ ]:
class SimplifiedGRD(nn.Module):
    """GRD with only reward mask and reward module — no dynamics."""

    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 64) -> None:
        super().__init__()
        self.psi_sr  = nn.Parameter(torch.zeros(state_dim,  2))
        self.psi_ar  = nn.Parameter(torch.zeros(action_dim, 2))
        self.rew_net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def predict_return(
        self,
        states: torch.Tensor,   # (batch, T, d_s)
        actions: torch.Tensor,  # (batch, T, d_a)
        tau: float = 1.0,
    ) -> torch.Tensor:          # (batch,)
        mask_sr = F.gumbel_softmax(self.psi_sr, tau=tau, dim=-1)[..., 1]
        mask_ar = F.gumbel_softmax(self.psi_ar, tau=tau, dim=-1)[..., 1]
        inp = torch.cat([states * mask_sr, actions * mask_ar], dim=-1)
        return self.rew_net(inp).squeeze(-1).sum(dim=1)  # sum over timesteps

    def hard_masks(self) -> tuple:
        c_sr = (self.psi_sr[:, 1] >= self.psi_sr[:, 0]).float()
        c_ar = (self.psi_ar[:, 1] >= self.psi_ar[:, 0]).float()
        return c_sr, c_ar


model_simple = SimplifiedGRD(STATE_DIM, ACTION_DIM)
opt_simple   = optim.Adam(model_simple.parameters(), lr=3e-4)
lam_sr, lam_ar = 5e-3, 5e-3

losses_simple = []
for step_idx in range(2000):
    opt_simple.zero_grad()
    ep_idx = torch.randint(N_EPISODES, (32,))
    pred_R = model_simple.predict_return(states[ep_idx], actions[ep_idx])
    probs  = F.softmax(model_simple.psi_sr, dim=-1)[:, 1]
    probs_a = F.softmax(model_simple.psi_ar, dim=-1)[:, 1]
    reg    = (
        lam_sr * (-torch.log(1 - probs + 1e-8)).sum() +
        lam_ar * (-torch.log(1 - probs_a + 1e-8)).sum()
    )
    loss = F.mse_loss(pred_R, returns[ep_idx]) + reg
    loss.backward()
    opt_simple.step()
    losses_simple.append(loss.item())

c_sr_s, c_ar_s = model_simple.hard_masks()
print("Simplified GRD (reward mask only):")
print(f"  C^{{s→r}} learned : {c_sr_s.int().tolist()}")
print(f"  C^{{s→r}} true    : {TRUE_C_SR.int().tolist()}")
print(f"  C^{{a→r}} learned : {c_ar_s.int().tolist()}")
print(f"  C^{{a→r}} true    : {TRUE_C_AR.int().tolist()}")
print(f"\nTotal parameters: {count_parameters(model_simple):,}")

### 11.2 Replace Gumbel-Softmax with straight-through binary

The straight-through estimator (STE) uses a hard 0/1 forward pass but passes gradients
as if the mask were continuous. This is faster (no Gumbel noise) and can work well when
the masks are expected to be very sparse:

In [ ]:
def straight_through_bernoulli(logits: torch.Tensor) -> torch.Tensor:
    """Hard 0/1 forward; sigmoid gradient backward (STE)."""
    probs = torch.sigmoid(logits)
    hard  = (probs > 0.5).float()
    # Straight-through: gradient flows through probs, not through the hard threshold
    return hard - probs.detach() + probs


# Compare Gumbel-Softmax vs STE gradient norms for a single mask entry
psi_test = nn.Parameter(torch.tensor([0.0, 0.0]))

# Gumbel
loss_gs = (1 - F.gumbel_softmax(psi_test, tau=1.0, dim=-1)[1]) ** 2
loss_gs.backward()
print(f"Gumbel-Softmax gradient on ψ: {psi_test.grad.tolist()}")

psi_test2 = nn.Parameter(torch.tensor([0.0, 0.0]))
# STE — apply to p = sigmoid(ψ₁ - ψ₀)
psi_scalar = psi_test2[1] - psi_test2[0]
mask_ste   = straight_through_bernoulli(psi_scalar)
loss_ste   = (1 - mask_ste) ** 2
loss_ste.backward()
print(f"STE gradient on ψ           : {psi_test2.grad.tolist()}")

### 11.3 Replace per-edge parameters with input-conditioned attention

Instead of learning *global* masks (same $C$ for all states), an **attention mechanism**
can produce *state-conditioned* soft masks:

$$C^{s\to r}(s_t) = \sigma\!\left(W_\text{attn} \cdot s_t\right)$$

This allows the relevant dimensions to vary with the current state — useful for
non-stationary environments, at the cost of losing the clean causal interpretation
(the mask is now a function, not a fixed graph structure).

### 11.4 Summary of simplifications

| Simplification | What you give up | When to use |
|---|---|---|
| Drop dynamics module | $C^{s\to s}$, $C^{a\to s}$; compact rep. via transitivity | When you only need the reward mask |
| Straight-through instead of Gumbel | Slightly noisier gradient estimates | When masks are expected to be very sparse |
| L1 regularisation instead of log-prior | Exact Bayesian interpretation | For simpler implementation |
| Fixed temperature $\tau=1$ | Optimal annealing schedule | When tuning $\tau$ is expensive |
| Attention mask (input-conditioned) | Global causal interpretation | Non-stationary or partial observability |
| Factored masks per dimension | Shared structure across dimensions | Very high-dimensional state |

---
## 12. Summary

```
Core idea:
  Reward redistribution can be framed as causal structure discovery.
  Learn binary masks C^{s→r}, C^{a→r} that select which inputs drive the reward.

Bayesian framing:
  Each mask entry ~ Bernoulli(p); prior pushes p → 0 (sparsity).
  Data likelihood (return constraint) pulls p → 1 for relevant dimensions.
  MAP = balance between sparsity and explanatory power.

Making it trainable:
  Gumbel-Softmax relaxes discrete {0,1} to continuous [0,1] during training.
  Greedy argmax recovers hard masks at inference.

Three losses:
  L_rew  — sum of predicted rewards equals observed return
  L_dyn  — next state predicted from masked (s, a)
  L_reg  — cross-entropy sparsity prior on all four masks

Compact representation:
  OR-closure of C^{s→r} over C^{s→s} gives s^min — the minimal
  sufficient state for policy learning.

Key challenges:
  - Identifiability only proved for linear Gaussian; nonlinear case is empirical
  - λ requires per-task tuning; no principled selection
  - Joint learning can be unstable; no stabilisation mechanism provided
  - O(d_s²) causal parameters scale poorly beyond ~400-dim state

Simplifications:
  - Drop dynamics module if compact rep is not needed
  - Straight-through estimator instead of Gumbel-Softmax
  - L1 instead of log-prior for sparsity
```